# BirdCLEF 2026 — Exp05 Kaggle Submission

This is the Kaggle-ready version of the local `exp05_submit` pipeline.

It trains the best local Perch stack on the labeled training soundscape windows, then predicts `test_soundscapes` and writes `submission.csv`.

Expected Kaggle inputs:
- BirdCLEF 2026 competition data.
- A dataset containing `perch_v2_no_dft.onnx` and, if needed, an `onnxruntime` wheel.
- A dataset or model asset containing Perch `labels.csv`.

Output:
- `/kaggle/working/submission.csv`

In [1]:
# Kaggle input discovery and imports
import gc
import json
import os
import re
import subprocess
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import soundfile as sf
from scipy.special import expit as sigmoid
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
LOCAL_ROOT = Path.cwd().resolve()
if not (LOCAL_ROOT / "data").exists() and (LOCAL_ROOT.parent / "data").exists():
    LOCAL_ROOT = LOCAL_ROOT.parent
IS_KAGGLE = KAGGLE_INPUT.exists()
WORK_DIR = KAGGLE_WORKING if IS_KAGGLE else LOCAL_ROOT
CACHE_DIR = WORK_DIR / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

SR = 32_000
WIN_SECONDS = 5
WIN_SAMPLES = SR * WIN_SECONDS
N_WINDOWS = 12
BATCH_SIZE = 4
PCA_DIM = 64
MIN_POS = 3
BEST_C = 0.05
BEST_CLASS_WEIGHT = "balanced"
BEST_PROBE_BLEND = 0.25
PRIOR_BLEND = 0.10
PROXY_REDUCE = "max"

print("IS_KAGGLE:", IS_KAGGLE)
print("WORK_DIR:", WORK_DIR)
if IS_KAGGLE:
    print("/kaggle/input top-level:", sorted(p.name for p in KAGGLE_INPUT.iterdir())[:30])

IS_KAGGLE: False
WORK_DIR: /Users/jroessler/Development/kaggle/birdclef


In [2]:
def first_existing(candidates):
    for path in candidates:
        path = Path(path)
        if path.exists():
            return path
    return None

def search_one(root, patterns):
    root = Path(root)
    if not root.exists():
        return None
    for pattern in patterns:
        hits = sorted(root.rglob(pattern))
        if hits:
            return hits[0]
    return None

DATA_DIR = first_existing([
    "/kaggle/input/birdclef-2026",
    "/kaggle/input/competitions/birdclef-2026",
    LOCAL_ROOT / "data",
])
if DATA_DIR is None and IS_KAGGLE:
    DATA_DIR = search_one(KAGGLE_INPUT, ["train_soundscapes_labels.csv"])
    DATA_DIR = DATA_DIR.parent if DATA_DIR else None

if DATA_DIR is None:
    raise FileNotFoundError("Could not find BirdCLEF data directory with train_soundscapes_labels.csv")

PERCH_ONNX_PATH = first_existing([
    LOCAL_ROOT / "data/models/perch/perch_v2_no_dft.onnx",
])
if PERCH_ONNX_PATH is None and IS_KAGGLE:
    PERCH_ONNX_PATH = search_one(KAGGLE_INPUT, ["perch_v2_no_dft.onnx", "*.onnx"])

PERCH_LABELS_PATH = first_existing([
    LOCAL_ROOT / "data/models/perch_tf/assets/labels.csv",
])
if PERCH_LABELS_PATH is None and IS_KAGGLE:
    # Prefer an assets/labels.csv if multiple generic labels.csv files exist.
    label_hits = sorted(KAGGLE_INPUT.rglob("labels.csv"))
    asset_hits = [p for p in label_hits if "asset" in str(p).lower() or "perch" in str(p).lower()]
    PERCH_LABELS_PATH = (asset_hits or label_hits)[0] if label_hits else None

if PERCH_ONNX_PATH is None:
    raise FileNotFoundError("Attach a Kaggle dataset containing perch_v2_no_dft.onnx")
if PERCH_LABELS_PATH is None:
    raise FileNotFoundError("Attach a Kaggle dataset/model containing Perch assets/labels.csv")

# Use ONNX Runtime if it is already installed; otherwise install it only from an
# attached input wheel. Internet installs are not allowed for competition submissions.
try:
    import onnxruntime as ort
    USE_ONNX = True
    ONNX_WHL = None
    print("ONNX Runtime already available")
except Exception as import_error:
    ONNX_WHL = search_one(KAGGLE_INPUT if IS_KAGGLE else LOCAL_ROOT, ["onnxruntime-*.whl"])
    if ONNX_WHL is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", str(ONNX_WHL)], check=True)
        print("ONNX Runtime installed from attached wheel:", ONNX_WHL)
    else:
        print("WARNING: ONNX Runtime wheel not found. Attach a dataset such as rishikeshjani/perch-onnx-for-birdclef-2026.")
    try:
        import onnxruntime as ort
        USE_ONNX = True
        print("ONNX Runtime available")
    except Exception as exc:
        USE_ONNX = False
        print("ONNX Runtime unavailable:", repr(exc))
        raise ModuleNotFoundError(
            "onnxruntime is required. Attach a Kaggle input dataset containing an onnxruntime-*.whl file; "
            "internet installs are intentionally disabled."
        ) from exc

TRAIN_SOUNDSCAPE_DIR = DATA_DIR / "train_soundscapes"
TEST_SOUNDSCAPE_DIR = DATA_DIR / "test_soundscapes"
SUBMISSION_PATH = WORK_DIR / "submission.csv"
SUMMARY_PATH = CACHE_DIR / "exp05_kaggle_submit_summary.json"

print("DATA_DIR:", DATA_DIR)
print("PERCH_ONNX_PATH:", PERCH_ONNX_PATH)
print("PERCH_LABELS_PATH:", PERCH_LABELS_PATH)
print("onnxruntime:", ort.__version__)
print("TRAIN_SOUNDSCAPE_DIR exists:", TRAIN_SOUNDSCAPE_DIR.exists())
print("TEST_SOUNDSCAPE_DIR exists:", TEST_SOUNDSCAPE_DIR.exists())

ONNX Runtime already available
DATA_DIR: /Users/jroessler/Development/kaggle/birdclef/data
PERCH_ONNX_PATH: /Users/jroessler/Development/kaggle/birdclef/data/models/perch/perch_v2_no_dft.onnx
PERCH_LABELS_PATH: /Users/jroessler/Development/kaggle/birdclef/data/models/perch_tf/assets/labels.csv
onnxruntime: 1.23.2
TRAIN_SOUNDSCAPE_DIR exists: True
TEST_SOUNDSCAPE_DIR exists: True


In [3]:
taxonomy = pd.read_csv(DATA_DIR / "taxonomy.csv")
taxonomy["primary_label"] = taxonomy["primary_label"].astype(str)
taxon_ids = taxonomy["primary_label"].tolist()
taxon_to_idx = {tid: i for i, tid in enumerate(taxon_ids)}
class_name = taxonomy.set_index("primary_label")["class_name"].to_dict()
N_CLASSES = len(taxon_ids)

sample_sub = pd.read_csv(DATA_DIR / "sample_submission.csv")
assert sample_sub.columns[1:].astype(str).tolist() == taxon_ids, "taxonomy order differs from sample_submission"

perch_labels = pd.read_csv(PERCH_LABELS_PATH, header=0)
# Perch labels.csv usually has one scientific-name column. Keep the first column only.
perch_labels = perch_labels.iloc[:, [0]].copy()
perch_labels.columns = ["scientific_name"]
perch_labels["bc_index"] = np.arange(len(perch_labels))

mapping = taxonomy.merge(perch_labels, on="scientific_name", how="left")
comp_to_bc = {
    taxon_to_idx[str(row["primary_label"])] : int(row["bc_index"])
    for _, row in mapping[mapping["bc_index"].notna()].iterrows()
}

def time_to_sec(t):
    h, m, s = map(int, str(t).split(":"))
    return h * 3600 + m * 60 + s

def parse_any_soundscape_id(name):
    stem = Path(str(name)).stem
    m = re.search(r"_(S\d+)_([0-9]{8})_([0-9]{6})", stem)
    if not m:
        return {"site": "UNK", "hour_utc": -1, "month": -1}
    site, ymd, hms = m.groups()
    return {"site": site, "hour_utc": int(hms[:2]), "month": int(ymd[4:6])}

def parse_labels(label_str):
    vec = np.zeros(N_CLASSES, dtype=np.float32)
    for tid in str(label_str).split(";"):
        tid = tid.strip()
        if tid in taxon_to_idx:
            vec[taxon_to_idx[tid]] = 1.0
    return vec

raw_df = pd.read_csv(DATA_DIR / "train_soundscapes_labels.csv")
raw_df["start_sec"] = raw_df["start"].apply(time_to_sec)
labels_df = (
    raw_df
    .drop_duplicates(subset=["filename", "start_sec"])
    .sort_values(["filename", "start_sec"])
    .reset_index(drop=True)
)
labels_df = pd.concat([labels_df, labels_df["filename"].apply(parse_any_soundscape_id).apply(pd.Series)], axis=1)
Y = np.stack(labels_df["primary_label"].apply(parse_labels).values)

print(f"Competition classes: {N_CLASSES}")
print(f"Mapped Perch classes: {len(comp_to_bc)}/{N_CLASSES}")
print(f"Training labeled windows: {len(labels_df)} from {labels_df['filename'].nunique()} files")
print("Sample submission shape:", sample_sub.shape)

Competition classes: 234
Mapped Perch classes: 203/234
Training labeled windows: 739 from 66 files
Sample submission shape: (3, 235)


In [4]:
sess = ort.InferenceSession(str(PERCH_ONNX_PATH), providers=["CPUExecutionProvider"])
print("Perch input:", sess.get_inputs()[0].name, sess.get_inputs()[0].shape)
print("Perch outputs:", [o.name for o in sess.get_outputs()])

def run_perch_batch(waves):
    # Most exported notebooks use output names "embedding" and "label".
    try:
        emb, logits = sess.run(["embedding", "label"], {"inputs": waves})
    except Exception:
        outs = sess.run(None, {sess.get_inputs()[0].name: waves})
        emb, logits = outs[0], outs[1]
    return emb.astype(np.float32), logits.astype(np.float32)

def infer_labeled_train_windows():
    emb_cache = CACHE_DIR / "train_labeled_perch_embeddings.npy"
    log_cache = CACHE_DIR / "train_labeled_perch_logits.npy"
    if emb_cache.exists() and log_cache.exists():
        print("Loading cached train Perch outputs")
        return np.load(emb_cache).astype(np.float32), np.load(log_cache).astype(np.float32)

    all_emb, all_logits = [], []
    for fname, grp in tqdm(labels_df.groupby("filename"), desc="Train files"):
        audio_path = TRAIN_SOUNDSCAPE_DIR / fname
        if not audio_path.exists():
            raise FileNotFoundError(audio_path)
        audio, _ = sf.read(str(audio_path), dtype="float32", always_2d=False)
        chunks = []
        for _, row in grp.iterrows():
            start = int(row["start_sec"]) * SR
            chunk = audio[start : start + WIN_SAMPLES]
            if len(chunk) < WIN_SAMPLES:
                chunk = np.pad(chunk, (0, WIN_SAMPLES - len(chunk)))
            chunks.append(chunk.astype(np.float32))

        embs, logits = [], []
        for i in range(0, len(chunks), BATCH_SIZE):
            e, l = run_perch_batch(np.stack(chunks[i : i + BATCH_SIZE]))
            embs.append(e)
            logits.append(l)
        all_emb.append(np.concatenate(embs, axis=0))
        all_logits.append(np.concatenate(logits, axis=0))
        gc.collect()

    emb = np.concatenate(all_emb, axis=0).astype(np.float32)
    logits = np.concatenate(all_logits, axis=0).astype(np.float32)
    np.save(emb_cache, emb)
    np.save(log_cache, logits)
    print("Cached train Perch outputs:", emb.shape, logits.shape)
    return emb, logits

# Local dry-runs can reuse exp02 cache if train_soundscapes audio is absent.
local_emb_cache = LOCAL_ROOT / "data/cache/exp02_embeddings.npy"
local_log_cache = LOCAL_ROOT / "data/cache/exp02_logits.npy"
if not IS_KAGGLE and local_emb_cache.exists() and local_log_cache.exists():
    emb_train = np.load(local_emb_cache).astype(np.float32)
    logits_train = np.load(local_log_cache).astype(np.float32)
    print("Loaded local exp02 cache:", emb_train.shape, logits_train.shape)
else:
    emb_train, logits_train = infer_labeled_train_windows()

Perch input: inputs ['batch', 160000]
Perch outputs: ['embedding', 'spatial_embedding', 'spectrogram', 'label']


Loaded local exp02 cache: (739, 1536) (739, 14795)


In [5]:
def build_comp_scores(logits, proxy_reduce=PROXY_REDUCE):
    scores = np.zeros((len(logits), N_CLASSES), dtype=np.float32)
    for comp_idx, bc_idx in comp_to_bc.items():
        scores[:, comp_idx] = logits[:, bc_idx]

    proxy_map = {}
    if proxy_reduce == "none":
        return scores, proxy_map

    for _, row in mapping[mapping["bc_index"].isna()].iterrows():
        target = str(row["primary_label"])
        scientific_name = str(row["scientific_name"])
        genus = scientific_name.split()[0]
        hits = perch_labels[
            perch_labels["scientific_name"].astype(str).str.match(rf"^{re.escape(genus)}\s", na=False)
        ]
        if hits.empty:
            continue
        hit_idx = hits["bc_index"].astype(int).values
        comp_idx = taxon_to_idx[target]
        if proxy_reduce == "max":
            scores[:, comp_idx] = logits[:, hit_idx].max(axis=1)
        elif proxy_reduce == "mean":
            scores[:, comp_idx] = logits[:, hit_idx].mean(axis=1)
        else:
            raise ValueError(proxy_reduce)
        proxy_map[target] = {
            "target_scientific_name": scientific_name,
            "class_name": class_name.get(target),
            "genus": genus,
            "n_proxy_species": int(len(hit_idx)),
        }
    return scores, proxy_map

def fit_prior_tables(meta_df, y_train, strength_site=8.0, strength_hour=8.0, strength_site_hour=4.0):
    global_p = y_train.mean(axis=0).astype(np.float32)
    tables = {"global_p": global_p, "site": {}, "hour": {}, "site_hour": {}}
    for site, idx in meta_df.groupby("site").groups.items():
        idx = np.array(list(idx), dtype=int)
        p = (y_train[idx].sum(axis=0) + strength_site * global_p) / (len(idx) + strength_site)
        tables["site"][str(site)] = p.astype(np.float32)
    for hour, idx in meta_df.groupby("hour_utc").groups.items():
        idx = np.array(list(idx), dtype=int)
        p = (y_train[idx].sum(axis=0) + strength_hour * global_p) / (len(idx) + strength_hour)
        tables["hour"][int(hour)] = p.astype(np.float32)
    for (site, hour), idx in meta_df.groupby(["site", "hour_utc"]).groups.items():
        idx = np.array(list(idx), dtype=int)
        p = (y_train[idx].sum(axis=0) + strength_site_hour * global_p) / (len(idx) + strength_site_hour)
        tables["site_hour"][(str(site), int(hour))] = p.astype(np.float32)
    return tables

def prior_probs_from_tables(meta_df, tables):
    out = np.repeat(tables["global_p"][None, :], len(meta_df), axis=0).astype(np.float32)
    for i, row in enumerate(meta_df.itertuples(index=False)):
        site = str(row.site)
        hour = int(row.hour_utc)
        if hour in tables["hour"]:
            out[i] = 0.50 * out[i] + 0.50 * tables["hour"][hour]
        if site in tables["site"]:
            out[i] = 0.50 * out[i] + 0.50 * tables["site"][site]
        if (site, hour) in tables["site_hour"]:
            out[i] = 0.35 * out[i] + 0.65 * tables["site_hour"][(site, hour)]
    return np.clip(out, 1e-5, 1.0 - 1e-5)

scores_train, proxy_map = build_comp_scores(logits_train)
raw_train = sigmoid(scores_train)
scaler = StandardScaler()
pca = PCA(n_components=PCA_DIM, whiten=True, random_state=42)
X_train = np.concatenate([pca.fit_transform(scaler.fit_transform(emb_train)), scores_train], axis=1)

probes = {}
for c in tqdm(range(N_CLASSES), desc="Fit LR probes"):
    positives = Y[:, c].sum()
    if MIN_POS <= positives < len(Y):
        clf = LogisticRegression(C=BEST_C, class_weight=BEST_CLASS_WEIGHT, max_iter=1000, solver="lbfgs")
        clf.fit(X_train, Y[:, c])
        probes[c] = clf

prior_tables = fit_prior_tables(labels_df[["site", "hour_utc"]].reset_index(drop=True), Y)
print("Fitted probes:", len(probes))
print("Genus proxy targets:", len(proxy_map))

Fit LR probes:   0%|          | 0/234 [00:00<?, ?it/s]

Fitted probes: 64
Genus proxy targets: 6


In [6]:
def infer_test_soundscapes(paths):
    all_emb, all_logits, rows = [], [], []
    for path in tqdm(paths, desc="Test files"):
        audio, _ = sf.read(str(path), dtype="float32", always_2d=False)
        stem = path.stem
        parsed = parse_any_soundscape_id(path.name)
        chunks = []
        for end_sec in range(WIN_SECONDS, WIN_SECONDS * N_WINDOWS + 1, WIN_SECONDS):
            start = (end_sec - WIN_SECONDS) * SR
            chunk = audio[start : start + WIN_SAMPLES]
            if len(chunk) < WIN_SAMPLES:
                chunk = np.pad(chunk, (0, WIN_SAMPLES - len(chunk)))
            chunks.append(chunk.astype(np.float32))
            rows.append({"row_id": f"{stem}_{end_sec}", "site": parsed["site"], "hour_utc": parsed["hour_utc"]})

        embs, logits = [], []
        for i in range(0, len(chunks), BATCH_SIZE):
            e, l = run_perch_batch(np.stack(chunks[i : i + BATCH_SIZE]))
            embs.append(e)
            logits.append(l)
        all_emb.append(np.concatenate(embs, axis=0))
        all_logits.append(np.concatenate(logits, axis=0))
        gc.collect()

    return pd.DataFrame(rows), np.concatenate(all_emb).astype(np.float32), np.concatenate(all_logits).astype(np.float32)

def predict_from_perch_outputs(meta_df, emb, logits):
    scores, _ = build_comp_scores(logits)
    raw_probs = sigmoid(scores)
    X = np.concatenate([pca.transform(scaler.transform(emb)), scores], axis=1)
    probe_probs = raw_probs.copy()
    for c, clf in probes.items():
        probe_probs[:, c] = clf.predict_proba(X)[:, 1]
    tuned_probe = BEST_PROBE_BLEND * probe_probs + (1.0 - BEST_PROBE_BLEND) * raw_probs
    prior_probs = prior_probs_from_tables(meta_df[["site", "hour_utc"]].reset_index(drop=True), prior_tables)
    return np.clip((1.0 - PRIOR_BLEND) * tuned_probe + PRIOR_BLEND * prior_probs, 0.0, 1.0)

test_paths = sorted(TEST_SOUNDSCAPE_DIR.glob("*.ogg")) if TEST_SOUNDSCAPE_DIR.exists() else []
print("Test audio files:", len(test_paths))

if test_paths:
    test_meta, emb_test, logits_test = infer_test_soundscapes(test_paths)
    preds = predict_from_perch_outputs(test_meta, emb_test, logits_test)
    pred_df = pd.DataFrame(preds, columns=taxon_ids)
    pred_df.insert(0, "row_id", test_meta["row_id"].values)

    submission = sample_sub[["row_id"]].merge(pred_df, on="row_id", how="left")
    missing = submission[taxon_ids].isna().any(axis=1).sum()
    if missing:
        print(f"Warning: {missing} rows missing from audio inference; filling with global priors")
    for col in taxon_ids:
        submission[col] = submission[col].fillna(prior_tables["global_p"][taxon_to_idx[col]])
else:
    # This branch is only for local/debug runs or Kaggle's tiny visible sample if test audio is absent.
    print("No test audio found. Writing prior-only dry-run submission with sample_submission shape.")
    dry_meta = pd.DataFrame({"row_id": sample_sub["row_id"]})
    parsed = dry_meta["row_id"].str.rsplit("_", n=1).str[0].apply(parse_any_soundscape_id).apply(pd.Series)
    dry_meta = pd.concat([dry_meta, parsed], axis=1)
    preds = prior_probs_from_tables(dry_meta[["site", "hour_utc"]], prior_tables)
    submission = pd.DataFrame(preds, columns=taxon_ids)
    submission.insert(0, "row_id", sample_sub["row_id"].values)

assert submission.shape == sample_sub.shape, (submission.shape, sample_sub.shape)
assert submission.columns.tolist() == sample_sub.columns.tolist()
submission[taxon_ids] = submission[taxon_ids].clip(0.0, 1.0).astype(np.float32)
submission.to_csv(SUBMISSION_PATH, index=False)

summary = {
    "mode": "test_audio" if test_paths else "dry_run_no_test_audio",
    "submission_path": str(SUBMISSION_PATH),
    "shape": list(submission.shape),
    "test_files": len(test_paths),
    "fitted_probes": len(probes),
    "proxy_reduce": PROXY_REDUCE,
    "proxy_targets": len(proxy_map),
    "prior_blend": PRIOR_BLEND,
}
with open(SUMMARY_PATH, "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
submission.head()

Test audio files: 0
No test audio found. Writing prior-only dry-run submission with sample_submission shape.
{
  "mode": "dry_run_no_test_audio",
  "submission_path": "/Users/jroessler/Development/kaggle/birdclef/submission.csv",
  "shape": [
    3,
    235
  ],
  "test_files": 0,
  "fitted_probes": 64,
  "proxy_reduce": "max",
  "proxy_targets": 6,
  "prior_blend": 0.1
}


,row_id,1161364,116570,1176823,1491113,1595929,209233,22930,22956,22961,...,whnjay1,whtdov,whwpic1,y00678,yebcar,yebela1,yecmac,yecpar,yehcar1,yeofly1
0,BC2026_Test_0001_S05_20250227_010002_5,0.00001,0.009988,0.00001,0.162393,0.00001,0.00001,0.00001,0.00001,0.129355,...,0.00001,0.048405,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001
1,BC2026_Test_0001_S05_20250227_010002_10,0.00001,0.009988,0.00001,0.162393,0.00001,0.00001,0.00001,0.00001,0.129355,...,0.00001,0.048405,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001
2,BC2026_Test_0001_S05_20250227_010002_15,0.00001,0.009988,0.00001,0.162393,0.00001,0.00001,0.00001,0.00001,0.129355,...,0.00001,0.048405,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001,0.00001
